# Unsupervised Gaussian mixture workflow

This notebook replaces K-means with a **Gaussian Mixture Model (GMM)** for donor-transfer clustering.

The workflow is:
1. fit the **reference donor** in a shared standardized feature space
2. keep those learned component shapes fixed
3. assign the other donor against those same components
4. optionally adapt only the mixture weights for the non-reference donor

This is a better match than K-means when the hidden cell states are elliptical rather than round in feature space.

The clustering uses three features:
- **CD206 brightness**
- **Cell area**
- **Cell eccentricity**


In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "macrophage_analysis").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import macrophage_analysis as ma
from macrophage_analysis.analysis.clustering import (
    build_aligned_cluster_composition_table,
    ordered_present_values,
)
from macrophage_analysis.analysis.gaussian_mixture import run_split_gaussian_mixture
from macrophage_analysis.analysis.morphology_tables import build_morphology_table
from macrophage_analysis.notebook import (
    build_clustering_feature_labels,
    build_marker_dropdown,
)
from macrophage_analysis.plotting.clustering import (
    plot_aligned_cluster_composition,
    plot_cluster_feature_pairs,
)

gmm_donors = list(ma.DEFAULT_DONORS)
gmm_conditions = list(ma.DEFAULT_CONDITIONS)
gmm_antibody_dropdown = build_marker_dropdown()
gmm_feature_columns = ("intensity", "area", "eccentricity")
gmm_feature_label_extras = {
    "area": "Area",
    "eccentricity": "Eccentricity",
}
gmm_centroid_columns = (
    "cd206_intensity_centroid",
    "area_centroid",
    "eccentricity_centroid",
)
gmm_n_tiles = (8, 8)
gmm_pairplot_max_points_per_donor = 3000
display(gmm_antibody_dropdown)
gmm_donors, gmm_conditions, gmm_antibody_dropdown.value, gmm_feature_columns


Run this once. The extraction stays focused on **CD206**, because brightness, area, and eccentricity all come from the same single-cell measurement path.


In [ ]:
gmm_antibody = str(gmm_antibody_dropdown.value)
gmm_feature_labels = build_clustering_feature_labels(
    gmm_antibody,
    extra_labels=gmm_feature_label_extras,
)

gmm_results = ma.extract_single_cell_fluorescence(
    donors=gmm_donors,
    conditions=gmm_conditions,
    antibody_order=[gmm_antibody],
    n_tiles=gmm_n_tiles,
)


Build the per-cell feature table. This is the same morphology table used in the K-means notebooks, but here it feeds the GMM instead.


In [ ]:
gmm_cells = build_morphology_table(
    gmm_results,
    antibody=gmm_antibody,
)

display(
    gmm_cells.groupby(["donor_label", "condition_label"], observed=True)
    .agg(
        cell_count=("cell_index", "count"),
        median_intensity=("intensity", "median"),
        median_area=("area", "median"),
        median_eccentricity=("eccentricity", "median"),
    )
    .reset_index()
)


## GMM parameters

Choose the number of components and the **reference donor**. The reference donor is fit fully. The other donor is then scored against those same learned Gaussian components, while the donor-specific mixture weights may still adapt.


In [ ]:
gmm_n_components = 3
gmm_random_seed = 222
gmm_max_iterations = 100
gmm_tolerance = 1e-4
gmm_reg_covar = 1e-6
gmm_non_reference_mode = "independent"
gmm_max_alignment_distance = 2.0
gmm_adapt_non_reference_weights = True

gmm_reference_donor_options = ordered_present_values(gmm_cells["donor_label"])
gmm_reference_donor_dropdown = widgets.Dropdown(
    options=gmm_reference_donor_options,
    value=gmm_reference_donor_options[0],
    description="Reference donor",
)
display(gmm_reference_donor_dropdown)


Run the donor-transfer GMM here. The component table shows the fixed reference centroids plus donor-specific assignment fractions and mean posterior confidence.


In [ ]:
gmm_reference_donor_label = str(gmm_reference_donor_dropdown.value)

(
    gmm_donor_feature_data,
    gmm_donor_runs,
    gmm_clustered_cells,
    gmm_component_table,
) = run_split_gaussian_mixture(
    gmm_cells,
    n_components=gmm_n_components,
    random_seed=gmm_random_seed,
    reference_donor_label=gmm_reference_donor_label,
    feature_columns=gmm_feature_columns,
    centroid_column_names=gmm_centroid_columns,
    max_iterations=gmm_max_iterations,
    tolerance=gmm_tolerance,
    reg_covar=gmm_reg_covar,
    adapt_non_reference_weights=gmm_adapt_non_reference_weights,
    non_reference_mode=gmm_non_reference_mode,
    max_alignment_distance=gmm_max_alignment_distance,
)

display(
    gmm_component_table[[
        "reference_donor_label",
        "donor_label",
        "cluster",
        "aligned_cluster",
        "alignment_distance",
        "reference_weight",
        "donor_weight",
        "assignment_cell_count",
        "assignment_fraction",
        "mean_assignment_probability",
        "cd206_intensity_centroid",
        "area_centroid",
        "eccentricity_centroid",
    ]].round({
        "alignment_distance": 3,
        "reference_weight": 3,
        "donor_weight": 3,
        "assignment_fraction": 3,
        "mean_assignment_probability": 3,
        "cd206_intensity_centroid": 2,
        "area_centroid": 1,
        "eccentricity_centroid": 3,
    })
)


## Pair plots

These are the main visual outputs. The GMM is fit in **3D**, but the results are displayed as donor-specific **pair plots** for every 2D projection, colored by the aligned cluster ID.


In [ ]:
plot_cluster_feature_pairs(
    gmm_clustered_cells,
    feature_columns=gmm_feature_columns,
    feature_labels=gmm_feature_labels,
    cluster_column="aligned_cluster",
    max_points_per_donor=gmm_pairplot_max_points_per_donor,
)


## Confidence QC

This table shows whether the donor-transfer assignments are actually confident. If the mean assignment probabilities are low, the data may be more continuous than cluster-like.


In [ ]:
display(
    gmm_clustered_cells.groupby(["donor_label", "condition_label", "aligned_cluster"], observed=True)
    .agg(
        cell_count=("assignment_probability", "size"),
        mean_assignment_probability=("assignment_probability", "mean"),
        median_assignment_probability=("assignment_probability", "median"),
    )
    .reset_index()
    .round({
        "mean_assignment_probability": 3,
        "median_assignment_probability": 3,
    })
)


## Treatment composition

As in the clustering notebooks, this shows how the unsupervised reference-aligned clusters are distributed across treatments.


In [ ]:
gmm_cluster_composition = build_aligned_cluster_composition_table(gmm_clustered_cells)

display(gmm_cluster_composition)
plot_aligned_cluster_composition(
    gmm_cluster_composition,
    reference_donor_label=gmm_reference_donor_label,
    cluster_count=gmm_n_components,
)
